<a href="https://colab.research.google.com/github/rameshaditya-me/Easy-Classical-ML-DL/blob/main/src/notebooks/discrete-regression/discrete-regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generalized Linear Models
---

**Generalized Linear Models (GLMs)** extend ordinary linear regression to many kinds of response variable. The same linear structure in the predictors can model continuous, discrete, binary, ordinal, categorical, and count outcomes — the choice of **link function** determines how $\beta^\top X$ connects to the distribution of $Y$.

The key idea is twofold. First, a link $g$ maps the linear predictor $\beta^\top X$ to a parameter of the outcome distribution (e.g. the mean). Second, the variance of each observation may depend on that predicted value — heteroscedasticity is built into the model rather than treated as a nuisance.

A GLM has three parts:

1. **Random component.** The outcome $Y$ is drawn from an **exponential-family** distribution (Gaussian, Bernoulli, Poisson, etc.).

2. **Systematic component.** One or more covariates $X_0, \ldots, X_d$ enter linearly through $\beta^\top X$.

3. **Link function.** A function $g$ connects the systematic component to the mean (or natural parameter) of the random component: $g(\mathbb{E}[Y \mid X]) = \beta^\top X$. A link function is called a canonical-link-function when it is directly equal to the natural parameter $\eta{\theta}$ of the exponential family.

### Exponential Family of Probability Distributions
---
Any probability distribution that can be written in the following parametric form belongs to the **exponential family**:

$$
p_{\theta}(y) = h(y)\,\exp\!\left(\eta(\theta)\,T(y) - A(\theta)\right)
$$

where $h(y)$ is the **base measure**, $\eta(\theta)$ is the **natural parameter**, $T(y)$ is the **sufficient statistic**, and $A(\theta)$ is the **log-partition function**.

#### Gaussian with unknown mean and known variance

$$
p(y) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left(-\frac{(y-\mu)^2}{2\sigma^2}\right)
$$

Expanding $(y-\mu)^2$ and matching the exponential-family form:

$$
p(y) =
\underset{\text{base measure}}{\overbrace{
  \frac{1}{\sqrt{2\pi\sigma^2}}\,
  \exp\!\left(-\dfrac{y^2}{2\sigma^2}\right)
}^{h(y)}}\;
\exp\!\left(
  \underset{\text{natural parameter}}{\overbrace{\dfrac{\mu}{\sigma^2}}^{\eta(\theta)}}\,
  \underset{\text{sufficient statistic}}{\overbrace{y}^{T(y)}}
  -
  \underset{\text{log-partition}}{\overbrace{\dfrac{\mu^2}{2\sigma^2}}^{A(\theta)}}
\right)
$$

Here $\theta = \mu$, so $\eta(\mu) = \mu/\sigma^2$ and $T(y) = y$.

#### Binomial distribution with a known number of trials $n$

Let $Y \in \{0, 1, \ldots, n\}$ count the number of successes in $n$ independent trials, each with success probability $p$:

$$
P(y) = \binom{n}{y}\, p^{y}(1-p)^{n-y}
$$

Rewriting in log form and collecting terms in $y$:

$$
P(y) = \binom{n}{y}\exp\!\left(y\log\frac{p}{1-p} + n\log(1-p)\right)
$$

Matching the exponential-family form $p_{\theta}(y) = h(y)\exp(\eta(\theta)T(y) - A(\theta))$:

$$
P(y) =
\underset{\text{base measure}}{\overbrace{
  \binom{n}{y}
}^{h(y)}}\;
\exp\!\left(
  \underset{\text{natural parameter}}{\overbrace{\log\!\left(\dfrac{p}{1-p}\right)}^{\eta(\theta)}}\,
  \underset{\text{sufficient statistic}}{\overbrace{y}^{T(y)}}
  -
  \underset{\text{log-partition}}{\overbrace{\left(-n\log(1-p)\right)}^{A(\theta)}}
\right)
$$

Here $\theta = p$, so $\eta(p) = \log\!\big(p/(1-p)\big)$ (the **log-odds**) and $T(y) = y$. Equivalently, writing $A$ in terms of the natural parameter alone:

$$
A(\eta) = n\log\!\left(1 + e^{\eta}\right)
$$

The **Bernoulli** distribution is the special case $n = 1$; logistic regression uses this exponential-family form with the logit link $\eta = \beta^\top x$.

### Choosing a GLM
---

A GLM is specified by three linked choices: the **type of response** you observe, the **exponential-family distribution** that models $Y$, and the **link function** $g$ that connects the linear predictor $\beta^\top X$ to a parameter of that distribution.

Ordinary linear regression is the GLM with $Y \sim \mathcal{N}(\mu, \sigma^2)$ and the **identity** link $g(\mu) = \mu$. Logistic regression is the GLM with $Y \sim \text{Bernoulli}(p)$ and the **logit** link $g(p) = \log\!\big(p/(1-p)\big)$. Every other exponential-family member — Poisson, Gamma, Negative Binomial, and so on — defines its own regression model once paired with a link, usually the **canonical link** that sets the natural parameter $\eta$ equal to $\beta^\top X$.

**Step1:** Given a dataset $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^n$, the first step in fitting a GLM is to identify the response type and select the matching family and link:

| Nature of data | Target family | Canonical link $g(\mu)$ |
| --- | --- | --- |
| Continuous in $\mathbb{R}$ | $Y \sim \mathcal{N}(\mu, \sigma^2)$ | Identity: $\beta^\top X = \mu$ |
| Binary in $\{0, 1\}$ | $Y \sim \text{Bernoulli}(p)$ | Logit: $\beta^\top X = \log\!\dfrac{p}{1-p}$ |
| Counts in $\{0, 1, 2, \ldots\}$ | $Y \sim \text{Poisson}(\lambda)$ | Log: $\beta^\top X = \log \lambda$ |
| Counts with overdispersion | $Y \sim \text{NegBinomial}(r, p)$ | Log: $\beta^\top X = \log \mu$ |
| Positive continuous | $Y \sim \text{Gamma}(\alpha, \beta)$ | Inverse: $\beta^\top X = 1/\mu$ |
| Positive continuous (skewed) | $Y \sim \text{Exponential}(\lambda)$ | Log: $\beta^\top X = \log \lambda$ | 

**Step 2:** Estimate the regression coefficients $\beta$ by **maximum likelihood** — find the $\beta$ that maximizes the likelihood of the observed $y_i$ under the chosen exponential-family distribution (equivalently, minimizes the deviance).

**Step 3:** To predict at a new input $x$, evaluate $\hat\beta^\top x$ to obtain the natural parameter $\hat\eta$, which fully specifies the fitted distribution. In practice we usually report the predicted mean $\mathbb{E}[Y \mid x]$, but the distributional form also lets us construct **prediction intervals** or **confidence bands** around $\hat{y}$.

### Confidence Intervals
---

Once a GLM is fitted, a point prediction at a new input $x$ is the estimated mean response

$$
\hat\mu = g^{-1}(\hat\beta^\top x),
$$

where $g$ is the link function. To quantify uncertainty, we distinguish two targets:

- A **confidence interval (CI) for the mean** $\mathbb{E}[Y \mid x]$ — how precisely we have estimated the fitted response at $x$.
- A **prediction interval (PI) for a new observation** $Y_{\text{new}}$ — the range a future outcome at $x$ is expected to fall in, including irreducible randomness in $Y$ itself.

To build either interval we need (i) a point estimate $\hat\beta$, (ii) a measure of how uncertain that estimate is, and (iii) a rule for turning estimate $\pm$ uncertainty into an interval. The three ideas below supply exactly that.

#### Fisher information

Let $\ell(\beta) = \log p(y \mid X, \beta)$ be the log-likelihood of the GLM. The **Fisher information matrix** quantifies how much a sample tells us about $\beta$:

$$
\mathcal{I}(\beta) = -\mathbb{E}\!\left[\frac{\partial^2 \ell(\beta)}{\partial \beta \,\partial \beta^\top}\right].
$$

Large curvature of $\ell$ (large $\mathcal{I}$) means the likelihood is sharply peaked and $\beta$ is tightly pinned down; flat curvature means more uncertainty. For large samples, the MLE satisfies the approximate normality result

$$
\hat\beta \;\approx\; \mathcal{N}\!\left(\beta,\; \mathcal{I}(\hat\beta)^{-1}\right),
$$



In [ ]:
#### Fisher information grows with sample size

Consider a Gaussian GLM with $\beta = (\beta_0, \beta_1)^\top$, identity link, and known $\sigma^2$. For design matrix $X \in \mathbb{R}^{n \times 2}$,

$$
\mathcal{I}(\beta) = \frac{1}{\sigma^2}\, X^\top X,
$$

so each new observation adds curvature to $\ell(\beta)$. The widget below plots **precomputed** log-likelihood contours on the $(\beta_0, \beta_1)$ plane for increasing $n$. Sharper, tighter contours correspond to larger $\det(\mathcal{I})$ — the likelihood concentrates more tightly around $\hat\beta$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, SelectionSlider
from IPython.display import display

# --- fixed data-generating process ---
rng = np.random.default_rng(42)
TRUE_BETA = np.array([1.0, 2.0])  # (intercept, slope)
SIGMA = 1.0
MAX_N = 200

x_full = rng.uniform(-2, 2, MAX_N)
X_full = np.column_stack([np.ones(MAX_N), x_full])
y_full = X_full @ TRUE_BETA + rng.normal(0, SIGMA, MAX_N)

# --- β grid for contour plots ---
b0_grid = np.linspace(-1.0, 3.0, 100)
b1_grid = np.linspace(0.0, 4.0, 100)
B0, B1 = np.meshgrid(b0_grid, b1_grid)


def log_likelihood_grid(B0, B1, X, y, sigma):
    """Gaussian log-likelihood evaluated on a 2-D β grid."""
    ll = np.zeros_like(B0, dtype=float)
    for i in range(len(y)):
        mu = B0 + B1 * X[i, 1]
        ll += -0.5 * ((y[i] - mu) / sigma) ** 2
    ll -= len(y) * np.log(sigma * np.sqrt(2 * np.pi))
    return ll


def fisher_information(X, sigma):
    return (X.T @ X) / sigma**2


# --- precompute contours and Fisher information for each sample size ---
SAMPLE_SIZES = [5, 10, 20, 40, 80, 120, 160, MAX_N]

precomputed = {}
for n in SAMPLE_SIZES:
    X_n = X_full[:n]
    y_n = y_full[:n]
    Z = log_likelihood_grid(B0, B1, X_n, y_n, SIGMA)
    beta_hat = np.linalg.lstsq(X_n, y_n, rcond=None)[0]
    I = fisher_information(X_n, SIGMA)
    precomputed[n] = {
        "Z": Z,
        "Z_rel": Z - Z.max(),  # peak at 0 for comparable contour levels
        "beta_hat": beta_hat,
        "I": I,
        "det_I": np.linalg.det(I),
    }

LEVELS = np.linspace(-35, 0, 14)


def show_likelihood(n):
    data = precomputed[n]
    fig, ax = plt.subplots(figsize=(7.5, 5.5))

    cf = ax.contourf(B0, B1, data["Z_rel"], levels=LEVELS, cmap="viridis", alpha=0.85)
    cs = ax.contour(B0, B1, data["Z_rel"], levels=LEVELS, colors="k", linewidths=0.4, alpha=0.5)
    fig.colorbar(cf, ax=ax, label=r"$\ell(\beta) - \max_\beta \ell(\beta)$")

    ax.plot(*TRUE_BETA, "r*", markersize=15, label=r"true $\beta$", zorder=5)
    ax.plot(*data["beta_hat"], "o", color="white", markeredgecolor="black",
            markersize=9, label=rf"MLE $\hat\beta$", zorder=5)

    ax.set_xlabel(r"$\beta_0$ (intercept)")
    ax.set_ylabel(r"$\beta_1$ (slope)")
    ax.set_title(
        rf"$n = {n}$ samples  |  "
        rf"$\det(\mathcal{{I}}) = {data['det_I']:.1f}$  |  "
        rf"$\hat\beta = ({data['beta_hat'][0]:.2f},\; {data['beta_hat'][1]:.2f})$"
    )
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


slider = SelectionSlider(
    options=[(f"n = {n}", n) for n in SAMPLE_SIZES],
    value=SAMPLE_SIZES[0],
    description="Samples",
    continuous_update=False,
)

ui = interact(show_likelihood, n=slider)
display(ui.widget)



so $\widehat{\mathrm{Cov}}(\hat\beta) \approx \mathcal{I}(\hat\beta)^{-1}$ is the standard large-sample covariance of $\hat\beta$.

#### IRLS and the information matrix $X^\top W X$

GLM coefficients are typically found by **iteratively reweighted least squares (IRLS)**. At each iteration the model is approximated by a weighted least-squares problem; at convergence the Fisher information takes the closed form

$$
\mathcal{I}(\hat\beta) = \frac{1}{\phi}\, X^\top W X,
$$

where $X$ is the design matrix, $\phi$ is the **dispersion parameter** ( $\phi = \sigma^2$ for Gaussian regression, $\phi = 1$ for Bernoulli and Poisson), and $W = \mathrm{diag}(w_1, \ldots, w_n)$ is a diagonal weight matrix with entries

$$
w_i = \frac{1}{\phi}\,\frac{1}{v(\mu_i)\, g'(\mu_i)^2}.
$$

Here $v(\mu)$ is the **variance function** of the chosen family (e.g. $v(\mu) = 1$ for Bernoulli, $v(\mu) = \mu$ for Poisson) and $g'(\mu)$ is the derivative of the link. In practice, statistical software returns $\widehat{\mathrm{Cov}}(\hat\beta) = (X^\top W X)^{-1}\phi$ directly from the fitted model.

#### Wald interval

If $\hat\theta$ is an approximately normal estimator with standard error $\mathrm{SE}(\hat\theta)$, a **Wald (normal-approximation) interval** at level $1 - \alpha$ is

$$
\hat\theta \;\pm\; z_{\alpha/2}\,\mathrm{SE}(\hat\theta),
$$

where $z_{\alpha/2}$ is the standard normal quantile (e.g. $1.96$ for 95%). When an extra nuisance parameter such as $\sigma^2$ is estimated from the data, $z_{\alpha/2}$ is replaced by $t_{\alpha/2}$ with the appropriate degrees of freedom.

---

**Putting it together.** With $\widehat{\mathrm{Cov}}(\hat\beta)$ in hand, the linear predictor $\hat\eta = \hat\beta^\top x$ has standard error

$$
\mathrm{SE}(\hat\eta) = \sqrt{x^\top \widehat{\mathrm{Cov}}(\hat\beta)\, x}.
$$

The general recipe is:

1. Form a **Wald interval** on the **link scale**: $\hat\eta \pm z_{\alpha/2}\,\mathrm{SE}(\hat\eta)$ (or use $t_{\alpha/2}$ when $\sigma^2$ must also be estimated, as in linear regression).
2. Map the endpoints back through $g^{-1}$ to obtain a CI for the mean $\mu$.
3. Add the observation-level variance $v(\mu)$ prescribed by the chosen family (e.g. $\sigma^2$ for Gaussian, $\mu$ for Poisson) to obtain a PI for a new $y$.

---

#### Example 1: Linear regression (Gaussian GLM, identity link)

Model: $Y \mid x \sim \mathcal{N}(\mu, \sigma^2)$ with $\mu = \beta^\top x$.

**Point prediction:** $\hat y = \hat\beta^\top x$.

**95% CI for the mean** $\mathbb{E}[Y \mid x]$:

$$
\hat y \;\pm\; t_{n-p-1,\;\alpha/2}\;\hat\sigma\;\sqrt{x^\top (X^\top X)^{-1} x},
$$

where $\hat\sigma^2$ is the residual mean square, $p$ is the number of coefficients, and $t_{n-p-1,\;\alpha/2}$ accounts for estimating $\sigma^2$.

**95% prediction interval** for a new observation $Y_{\text{new}}$ at the same $x$:

$$
\hat y \;\pm\; t_{n-p-1,\;\alpha/2}\;\hat\sigma\;\sqrt{1 + x^\top (X^\top X)^{-1} x}.
$$

The extra $+1$ under the square root is the contribution of the observation noise $\sigma^2$ — the PI is always wider than the CI for the mean.

---

#### Example 2: Logistic regression (Bernoulli GLM, logit link)

Model: $Y \mid x \sim \text{Bernoulli}(p)$ with $\log\!\dfrac{p}{1-p} = \beta^\top x$.

**Point prediction:** $\hat p = \sigma(\hat\beta^\top x) = \dfrac{1}{1 + e^{-\hat\beta^\top x}}$.

Because $p$ is bounded in $[0,1]$, we form the interval on the **log-odds scale** $\hat\eta = \hat\beta^\top x$ and transform back:

$$
\mathrm{SE}(\hat\eta) = \sqrt{x^\top \widehat{\mathrm{Cov}}(\hat\beta)\, x},
$$

$$
\hat p \;\in\; \left(\sigma\!\big(\hat\eta - z_{\alpha/2}\,\mathrm{SE}(\hat\eta)\big),\;\; \sigma\!\big(\hat\eta + z_{\alpha/2}\,\mathrm{SE}(\hat\eta)\big)\right).
$$

This is a **95% CI for the success probability** $\mathbb{E}[Y \mid x] = p$.

For a single Bernoulli outcome, $Y_{\text{new}} \in \{0, 1\}$, so a classical symmetric PI does not apply. Instead, the fitted $\hat p$ is the predictive probability $\mathbb{P}(Y_{\text{new}} = 1 \mid x)$, and uncertainty about that probability is captured by the CI above.